# Filtered Dense Search: `$eq`, `$in`, and `$range` in Practice

In this tutorial, you will:

- Combine semantic vector search with server-side metadata filters to build precise, production-ready retrieval pipelines.
- Run 15 filter configurations using all three of Endee's operators: `$eq`, `$in`, and `$range`
- Combine multiple filters with AND conditions
- See exactly how each filter changes the ranked result set

> **This notebook is a companion to the blog post *Filtered Dense Search: `$eq`, `$in`, and `$range` in Practice*.**  
> The cells below are unchanged from the original tutorial. The sections above and below the code provide the conceptual framing, implementation notes, and real-world patterns from the post.

---

## Why Filtered Search?

Pure semantic search answers one question: *which documents are most similar to this query?*  
Filtered semantic search answers a different, more useful question: *which documents, **among those I care about**, are most similar to this query?*

The distinction matters more than it looks. Without filters, a search for `"AI in healthcare"` across a multi-tenant product returns results from every user's data. With a filter on `tenant_id`, only the calling user's documents enter the ranking stage. The embedding model never had to learn tenant isolation — it just ranks within a pre-restricted pool.

```
┌─────────────────────────────────────────────────────────────┐
│                   Filtered Dense Search                      │
│                                                              │
│  Query ──► Embed ──► [ Filter Gate ] ──► HNSW Rank ──► Top-K│
│                            │                                 │
│                     $eq / $in / $range                       │
│                     (server-side, pre-vector)                │
└─────────────────────────────────────────────────────────────┘
```

**Key insight:** filters are *eligibility gates*, not ranking signals. A document that fails the filter never enters the ranking stage. A document that passes the filter but is semantically irrelevant will still rank at the bottom.

## The Three Operators

Endee supports three server-side filter operators that cover almost every real-world use case:

| Operator | What it does | Example |
|----------|-------------|---------|
| `$eq` | Exact match — string, bool, or int | `{"category": {"$eq": "tech"}}` |
| `$in` | List membership — OR *within* a field | `{"category": {"$in": ["health", "science"]}}` |
| `$range` | Inclusive numeric range `[start, end]` | `{"year": {"$range": [2022, 2024]}}` |

All three are evaluated **server-side**, before any vector ranking occurs. No `top_k=len(DOCUMENTS)` + Python post-filter needed.

**AND is the only multi-filter logic.** Multiple entries in the `filter` list are always ANDed:

```python
filter=[
    {"category": {"$eq":    "tech"}},       # must match
    {"year":     {"$range": [2022, 2024]}},  # AND must match
    {"premium":  {"$eq":    True}},          # AND must match
]
```

OR *across different fields* requires separate queries and client-side merging. OR *within* a single field is exactly what `$in` is for.

## The Upsert Contract

The most important rule: **you must declare every filterable field at upsert time.**

```python
index.upsert([{
    "id":     "doc_01",
    "vector": dense_model.encode(text).tolist(),
    "meta":   {"title": "Neural Nets & Vision", "category": "tech", ...},   # stored, not indexed
    "filter": {                    # ← index-time declaration
        "category": "tech",
        "year":     2023,
        "rating":   4.5,
        "author":   "alice",
        "premium":  True,
    },
}])
```

- **`meta`** — free-form dict returned with results. Not indexed, not filterable.  
- **`filter`** — declares the fields that can be queried at search time. A field missing here **cannot** be used in a filter later.

Think of `filter` as a column declaration — you cannot query a column that does not exist in the index.

---


## 1. Install dependencies

Install `endee` for the vector store and `sentence-transformers` for the embedding model.

In [1]:
# Uncomment if needed:
# !pip install endee sentence-transformers

from endee import Endee
from sentence_transformers import SentenceTransformer

print("✅ Imports successful!")

/opt/miniconda3/envs/tehr/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful!


## 2. Configure your index

Set the index name, model, embedding dimension, and similarity metric. You will reuse these constants throughout the tutorial.

| Parameter | Value | Why |
|-----------|-------|-----|
| `INDEX_NAME` | `dense_filter_demo` | Name of the Endee index |
| `DENSE_MODEL_NAME` | `all-MiniLM-L6-v2` | Fast 384-dim sentence encoder |
| `DENSE_DIM` | `384` | Must match the model output size |
| `SPACE_TYPE` | `cosine` | Similarity metric (higher = more similar) |

In [10]:
INDEX_NAME       = "dense_filter_demo"
DENSE_MODEL_NAME = "nomic-embed-text:latest"
DENSE_DIM        = 768   # nomic embeddings are 768-dim
SPACE_TYPE       = "cosine"

print(f"Index       : {INDEX_NAME}")
print(f"Model       : {DENSE_MODEL_NAME}")
print(f"Dimension   : {DENSE_DIM}")
print(f"Space type  : {SPACE_TYPE}")

Index       : dense_filter_demo
Model       : nomic-embed-text:latest
Dimension   : 768
Space type  : cosine


## 3. Define your document corpus

You will work with 20 research articles across four categories. Each article has five metadata fields you will use as filter dimensions:

| Field | Type | Values |
|-------|------|--------|
| `category` | string | `tech`, `science`, `health`, `business` |
| `year` | int | 2020–2024 |
| `rating` | float | 3.5–4.9 |
| `author` | string | `alice`, `bob`, `carol`, `dave` |
| `premium` | bool | `True` / `False` |

Distribution: 8 tech, 4 science, 4 health, 4 business articles.

In [3]:
DOCUMENTS = [
    # ── tech ──────────────────────────────────────────────────────────────────
    {"id": "doc_01", "text": "Neural networks are revolutionising image recognition and computer vision tasks",
     "meta": {"title": "Neural Nets & Vision",       "category": "tech",     "year": 2023, "rating": 4.5, "author": "alice", "premium": True}},
    {"id": "doc_02", "text": "Quantum computing promises exponential speedup for optimisation and cryptography",
     "meta": {"title": "Quantum Computing",          "category": "tech",     "year": 2024, "rating": 4.8, "author": "alice", "premium": True}},
    {"id": "doc_03", "text": "Vector databases power modern semantic search and retrieval-augmented generation",
     "meta": {"title": "Vector Databases",           "category": "tech",     "year": 2023, "rating": 4.3, "author": "carol", "premium": False}},
    {"id": "doc_04", "text": "Natural language processing transforms customer service and conversational AI",
     "meta": {"title": "NLP & Conversational AI",    "category": "tech",     "year": 2021, "rating": 3.9, "author": "carol", "premium": False}},
    {"id": "doc_05", "text": "Transformer architecture changed the landscape of deep learning and NLP",
     "meta": {"title": "Transformer Architecture",   "category": "tech",     "year": 2020, "rating": 4.9, "author": "alice", "premium": True}},
    {"id": "doc_06", "text": "Graph neural networks model complex relational and social network data",
     "meta": {"title": "Graph Neural Networks",      "category": "tech",     "year": 2022, "rating": 4.2, "author": "carol", "premium": True}},
    {"id": "doc_07", "text": "Autonomous vehicles use lidar sensors and deep learning for self-driving navigation",
     "meta": {"title": "Self-Driving Cars",          "category": "tech",     "year": 2024, "rating": 4.5, "author": "alice", "premium": True}},
    {"id": "doc_08", "text": "Federated learning enables privacy-preserving distributed model training across devices",
     "meta": {"title": "Federated Learning",         "category": "tech",     "year": 2021, "rating": 4.3, "author": "alice", "premium": True}},
    # ── science ───────────────────────────────────────────────────────────────
    {"id": "doc_09", "text": "CRISPR gene editing opens new frontiers in personalised medicine and genetic therapy",
     "meta": {"title": "CRISPR Gene Editing",        "category": "science",  "year": 2022, "rating": 4.7, "author": "bob",   "premium": True}},
    {"id": "doc_10", "text": "Climate change research leverages satellite imagery and AI-driven climate models",
     "meta": {"title": "AI Climate Models",          "category": "science",  "year": 2023, "rating": 4.1, "author": "alice", "premium": False}},
    {"id": "doc_11", "text": "Precision agriculture uses IoT sensors and machine learning to optimise crop yields",
     "meta": {"title": "Precision Agriculture",      "category": "science",  "year": 2022, "rating": 4.0, "author": "carol", "premium": False}},
    {"id": "doc_12", "text": "Microbiome research reveals the gut-brain connection and its role in mental health",
     "meta": {"title": "Gut-Brain Axis",             "category": "science",  "year": 2021, "rating": 3.9, "author": "bob",   "premium": False}},
    # ── health ────────────────────────────────────────────────────────────────
    {"id": "doc_13", "text": "Machine learning models predict patient outcomes and enable early disease detection",
     "meta": {"title": "ML in Healthcare",           "category": "health",   "year": 2022, "rating": 4.2, "author": "bob",   "premium": True}},
    {"id": "doc_14", "text": "Mental health apps provide accessible therapy tools and mindfulness training programmes",
     "meta": {"title": "Mental Health Apps",         "category": "health",   "year": 2023, "rating": 4.4, "author": "bob",   "premium": True}},
    {"id": "doc_15", "text": "Wearable fitness trackers improve personal health monitoring and chronic disease management",
     "meta": {"title": "Wearable Health Tech",       "category": "health",   "year": 2021, "rating": 3.8, "author": "dave",  "premium": False}},
    {"id": "doc_16", "text": "Telemedicine expands access to specialist healthcare in rural and underserved communities",
     "meta": {"title": "Telemedicine Access",        "category": "health",   "year": 2022, "rating": 4.1, "author": "bob",   "premium": False}},
    # ── business ──────────────────────────────────────────────────────────────
    {"id": "doc_17", "text": "Blockchain technology disrupts supply chain transparency and provenance tracking",
     "meta": {"title": "Blockchain Supply Chain",    "category": "business", "year": 2020, "rating": 3.5, "author": "dave",  "premium": False}},
    {"id": "doc_18", "text": "Renewable energy startups attract record venture capital investment in clean technology",
     "meta": {"title": "Clean Energy VC",            "category": "business", "year": 2024, "rating": 4.6, "author": "dave",  "premium": True}},
    {"id": "doc_19", "text": "Fintech companies reshape personal banking, payments, and embedded financial services",
     "meta": {"title": "Fintech Revolution",         "category": "business", "year": 2023, "rating": 3.7, "author": "dave",  "premium": False}},
    {"id": "doc_20", "text": "ESG investing criteria reshape corporate accountability and sustainable business strategy",
     "meta": {"title": "ESG Investing",             "category": "business", "year": 2022, "rating": 4.0, "author": "carol", "premium": True}},
]

print(f"📄 Corpus: {len(DOCUMENTS)} documents\n")
print(f"{'ID':<10} {'Category':<12} {'Year':<6} {'Rating':<8} {'Author':<8} {'Premium':<8} Title")
print("─" * 80)
for d in DOCUMENTS:
    m = d["meta"]
    print(f"{d['id']:<10} {m['category']:<12} {m['year']:<6} {m['rating']:<8} {m['author']:<8} {str(m['premium']):<8} {m['title']}")

📄 Corpus: 20 documents

ID         Category     Year   Rating   Author   Premium  Title
────────────────────────────────────────────────────────────────────────────────
doc_01     tech         2023   4.5      alice    True     Neural Nets & Vision
doc_02     tech         2024   4.8      alice    True     Quantum Computing
doc_03     tech         2023   4.3      carol    False    Vector Databases
doc_04     tech         2021   3.9      carol    False    NLP & Conversational AI
doc_05     tech         2020   4.9      alice    True     Transformer Architecture
doc_06     tech         2022   4.2      carol    True     Graph Neural Networks
doc_07     tech         2024   4.5      alice    True     Self-Driving Cars
doc_08     tech         2021   4.3      alice    True     Federated Learning
doc_09     science      2022   4.7      bob      True     CRISPR Gene Editing
doc_10     science      2023   4.1      alice    False    AI Climate Models
doc_11     science      2022   4.0      carol    

## 4. Load the embedding model

Load `all-MiniLM-L6-v2` to encode documents and queries into 384-dimensional vectors. The model is loaded once and reused for both upsert and search.

In [11]:
# print(f"Loading {DENSE_MODEL_NAME} ...")
# dense_model = SentenceTransformer(DENSE_MODEL_NAME)
# print(f"✅ Model loaded — output dim: {dense_model.get_sentence_embedding_dimension()}")
print(f"Loading {DENSE_MODEL_NAME} from Ollama ...")

import requests

def get_embedding(text: str):
    url = "http://localhost:11434/api/embeddings"
    
    payload = {
        "model": DENSE_MODEL_NAME,
        "prompt": text
    }
    
    response = requests.post(url, json=payload)
    response.raise_for_status()
    
    return response.json()["embedding"]

# test call
test_emb = get_embedding("test sentence")

print(f"✅ Model connected — output dim: {len(test_emb)}")

Loading nomic-embed-text:latest from Ollama ...
✅ Model connected — output dim: 768


## 5. Create your Endee index

Connect to Endee and create a cosine-similarity dense index. You need to do this before upserting any documents.

> `space_type="cosine"` causes the client to L2-normalise all vectors before storage. If the index already exists from a previous run, the conflict is silently ignored.
>
> Make sure Endee is running locally on `http://127.0.0.1:8080`.

In [13]:
print("Connecting to Endee ...")
client = Endee()
print("✅ Connected!")
client.set_base_url('http://localhost:8080/api/v1')

#client.set_base_url('http://localhost:1234/api/v1')

try:
    result = client.create_index(
        name=INDEX_NAME,
        dimension=DENSE_DIM,
        space_type=SPACE_TYPE,
    )
    print(f"✅ Index created: {result}")
except Exception as e:
    print(f"⚠️  {e} (index may already exist)")

index = client.get_index(INDEX_NAME)
print(f"📊 Index info: {index.describe()}")

Connecting to Endee ...
✅ Connected!
✅ Index created: Index created successfully
📊 Index info: {'name': 'dense_filter_demo', 'space_type': 'cosine', 'dimension': 768, 'sparse_model': 'None', 'is_hybrid': False, 'count': 0, 'precision': 'int8', 'M': 16, 'ef_con': 128}


## 6. Embed and index your documents

Encode each document and upsert it with two separate dicts:

- **`meta`** — any data you want returned with results. Not indexed, not filterable.
- **`filter`** — the fields you want to filter on at query time. **A field absent here cannot be used in a filter later** — treat it as a schema declaration.

In [6]:
print("Embedding and upserting documents ...\n")
payload = []

for doc in DOCUMENTS:
    vec = dense_model.encode(doc["text"]).tolist()
    m   = doc["meta"]
    payload.append({
        "id":     doc["id"],
        "vector": vec,
        "meta":   m,
        # All fields we may filter on must be declared here
        "filter": {
            "category": m["category"],
            "year":     m["year"],
            "rating":   m["rating"],
            "author":   m["author"],
            "premium":  m["premium"],
        },
    })
    print(f"  ✔ {doc['id']}  dim={len(vec)}")

result = index.upsert(payload)
print(f"\n✅ Upsert complete: {result}")

Embedding and upserting documents ...

  ✔ doc_01  dim=384
  ✔ doc_02  dim=384
  ✔ doc_03  dim=384
  ✔ doc_04  dim=384
  ✔ doc_05  dim=384
  ✔ doc_06  dim=384
  ✔ doc_07  dim=384
  ✔ doc_08  dim=384
  ✔ doc_09  dim=384
  ✔ doc_10  dim=384
  ✔ doc_11  dim=384
  ✔ doc_12  dim=384
  ✔ doc_13  dim=384
  ✔ doc_14  dim=384
  ✔ doc_15  dim=384
  ✔ doc_16  dim=384
  ✔ doc_17  dim=384
  ✔ doc_18  dim=384
  ✔ doc_19  dim=384
  ✔ doc_20  dim=384

✅ Upsert complete: Vectors inserted successfully


## 7. Run filtered queries

All 15 queries below use the same base text:

> **"AI applications in healthcare and medicine"**

This query has strong overlap with health documents and partial overlap with tech (ML, NLP), which makes filter effects easy to read in the output. The `show_results` helper prints a ranked table with id, score, and all metadata fields.

In [7]:
QUERY = "AI applications in healthcare and medicine"
query_vec = dense_model.encode(QUERY).tolist()
TOP_K = 5

def show_results(results, label=""):
    """Pretty-print query results."""
    header = f"  {'Rank':<5} {'ID':<10} {'Score':<8} {'Category':<12} {'Year':<6} {'Rating':<7} {'Author':<8} {'Premium':<8} Title"
    print(f"\n{'─'*len(header)}")
    if label:
        print(f"  {label}")
    print(header)
    print(f"  {'─'*120}")
    for i, r in enumerate(results, 1):
        m = r["meta"]
        print(
            f"  {i:<5} {r['id']:<10} {r['similarity']:<8.4f} "
            f"{m.get('category',''):<12} {m.get('year',''):<6} "
            f"{m.get('rating',''):<7} {m.get('author',''):<8} "
            f"{str(m.get('premium','')):<8} {m.get('title','')}"
        )
    print()

print(f'Base query : "{QUERY}"')
print(f"Top-K      : {TOP_K}")

Base query : "AI applications in healthcare and medicine"
Top-K      : 5


---
### 7.1 No Filter (Baseline)

Search across the **entire corpus** — every document is a candidate. This establishes the baseline ranking that all filtered queries below will diverge from.


In [8]:
results_all = index.query(vector=query_vec, top_k=TOP_K)
show_results(results_all, label="No filter — all 20 documents are candidates")


───────────────────────────────────────────────────────────────────────────────
  No filter — all 20 documents are candidates
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.4641   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_04     0.3194   tech         2021   3.9     carol    False    NLP & Conversational AI
  3     doc_01     0.3110   tech         2023   4.5     alice    True     Neural Nets & Vision
  4     doc_15     0.2622   health       2021   3.8     dave     False    Wearable Health Tech
  5     doc_16     0.2589   health       2022   4.1     bob      False    Telemedicine Access



**Response:** The top result should be the document most semantically aligned with your query. You may notice that documents from adjacent categories also appear in the top 5 — this is expected because embedding models cluster related concepts regardless of metadata labels. These baseline results are your ground truth: every filter below carves a subset of the full corpus, and ranking happens within that narrowed pool.


### 7.2  `$eq` — restrict to a single category

Use `$eq` to limit results to health articles only. Only documents whose `category` field equals `"health"` enter the candidate pool — the embedding model ranks semantically within that restricted set. Documents from other categories are excluded entirely regardless of their vector proximity.


In [9]:

results_health = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"category": {"$eq": "health"}}],
)
show_results(results_health, label='Filter: category == "health"  (4 candidates)')



───────────────────────────────────────────────────────────────────────────────
  Filter: category == "health"  (4 candidates)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_15     0.7386   health       2021   3.8     dave     False    Wearable Health Tech
  3     doc_16     0.7415   health       2022   4.1     bob      False    Telemedicine Access
  4     doc_14     0.7837   health       2023   4.4     bob      True     Mental Health Apps



**Response:** The top-ranked document from the health category should match the query semantically. You may notice the score value shifts compared to the unfiltered baseline — this is not a contradiction. When the filter pool is small (fewer than ~1,000 candidates), Endee switches from approximate HNSW search to exact brute-force distance computation, which reports cosine *distance* (lower = better) rather than cosine *similarity* (higher = better). The rank ordering is always correct regardless of which path is used. Do not compare raw score values across filtered and unfiltered queries — use rank positions instead.


### 7.3  `$eq` — switch to a different category

Run the same query with `category == "tech"` to see how rankings change when the pool shifts. Documents from other categories disappear entirely. The top result will now be the tech document whose embedding is closest to the query vector — which may differ from the baseline top result, illustrating that category boundaries do not always align with semantic proximity.


In [10]:
results_tech = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"category": {"$eq": "tech"}}],
)
show_results(results_tech, label='Filter: category == "tech"  (8 candidates)')



───────────────────────────────────────────────────────────────────────────────
  Filter: category == "tech"  (8 candidates)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_04     0.6804   tech         2021   3.9     carol    False    NLP & Conversational AI
  2     doc_01     0.6888   tech         2023   4.5     alice    True     Neural Nets & Vision
  3     doc_05     0.7530   tech         2020   4.9     alice    True     Transformer Architecture
  4     doc_06     0.7612   tech         2022   4.2     carol    True     Graph Neural Networks
  5     doc_07     0.8164   tech         2024   4.5     alice    True     Self-Driving Cars



### 7.4  `$eq` — filter by author

Restrict to `author == "bob"` to see only Bob's five articles. The embedding model ranks them by semantic relevance to the query — not by rating or category.

In [11]:
results_bob = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"author": {"$eq": "bob"}}],
)
show_results(results_bob, label='Filter: author == "bob"  (5 candidates: doc_09, doc_12, doc_13, doc_14, doc_16)')



───────────────────────────────────────────────────────────────────────────────
  Filter: author == "bob"  (5 candidates: doc_09, doc_12, doc_13, doc_14, doc_16)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_16     0.7415   health       2022   4.1     bob      False    Telemedicine Access
  3     doc_14     0.7837   health       2023   4.4     bob      True     Mental Health Apps
  4     doc_09     0.8833   science      2022   4.7     bob      True     CRISPR Gene Editing
  5     doc_12     0.9403   science      2021   3.9     bob      False    Gut-Brain Axis



### 7.5  `$eq` — boolean flag (premium content)

Pass `premium == True` to surface only premium articles. Free-tier and paid users can run the same query against the same index and receive a different result set, controlled entirely by this single filter.

In [12]:
results_premium = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"premium": {"$eq": True}}],
)
show_results(results_premium, label="Filter: premium == True  (11 candidates)")



───────────────────────────────────────────────────────────────────────────────
  Filter: premium == True  (11 candidates)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_01     0.6888   tech         2023   4.5     alice    True     Neural Nets & Vision
  3     doc_05     0.7530   tech         2020   4.9     alice    True     Transformer Architecture
  4     doc_06     0.7612   tech         2022   4.2     carol    True     Graph Neural Networks
  5     doc_14     0.7837   health       2023   4.4     bob      True     Mental Health Apps



### 7.6  AND — two `$eq` conditions

Multiple filters in the list are always **ANDed**. Combining `category == "health"` AND `premium == True` reduces the candidate pool to only documents satisfying both conditions simultaneously. The smaller the intersection, the fewer candidates the embedding model ranks over — and the more the score values may shift due to Endee's HNSW-vs-brute-force threshold.


In [13]:
results_health_premium = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"category": {"$eq": "health"}}, {"premium": {"$eq": True}}],
)
show_results(
    results_health_premium,
    label='Filter: category == "health" AND premium == True  (candidates: doc_13, doc_14)'
)



───────────────────────────────────────────────────────────────────────────────
  Filter: category == "health" AND premium == True  (candidates: doc_13, doc_14)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_14     0.7837   health       2023   4.4     bob      True     Mental Health Apps



### 7.7  AND — category + author

Restrict to a specific author's tech articles (`category == "tech"` AND `author == "alice"`). This demonstrates that metadata filters compose freely across different field types. The document with the highest rating in the filtered pool may rank last if its content is the least semantically relevant to the query — metadata attributes and embedding similarity are independent signals.


In [14]:
results_alice_tech = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"category": {"$eq": "tech"}}, {"author": {"$eq": "alice"}}],
)
show_results(
    results_alice_tech,
    label='Filter: category == "tech" AND author == "alice"  (candidates: doc_01, doc_02, doc_05, doc_07, doc_08)'
)



───────────────────────────────────────────────────────────────────────────────
  Filter: category == "tech" AND author == "alice"  (candidates: doc_01, doc_02, doc_05, doc_07, doc_08)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_01     0.6888   tech         2023   4.5     alice    True     Neural Nets & Vision
  2     doc_05     0.7530   tech         2020   4.9     alice    True     Transformer Architecture
  3     doc_07     0.8164   tech         2024   4.5     alice    True     Self-Driving Cars
  4     doc_08     0.8743   tech         2021   4.3     alice    True     Federated Learning
  5     doc_02     0.9531   tech         2024   4.8     alice    True     Quantum Computing



### 7.8  `$range` — numeric year window

`$range` takes a two-element array `[start, end]` (inclusive) and evaluates entirely server-side. No client-side post-filtering needed. Pass a year range to gate on recency.

In [15]:
results_year_range = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"year": {"$range": [2022, 2024]}}],
)
show_results(results_year_range, label="Filter: year in [2022, 2024]  (server-side $range)")


───────────────────────────────────────────────────────────────────────────────
  Filter: year in [2022, 2024]  (server-side $range)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_01     0.6888   tech         2023   4.5     alice    True     Neural Nets & Vision
  3     doc_16     0.7415   health       2022   4.1     bob      False    Telemedicine Access
  4     doc_10     0.7522   science      2023   4.1     alice    False    AI Climate Models
  5     doc_06     0.7612   tech         2022   4.2     carol    True     Graph Neural Networks



### 7.9  `$range` — rating quality floor

Use `rating ∈ [4.0, 5.0]` to enforce a minimum quality bar before ranking begins. Documents rated below 4.0 are excluded regardless of their semantic relevance.

In [16]:
results_high_rated = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"rating": {"$range": [4.0, 5.0]}}],
)
show_results(results_high_rated, label="Filter: rating in [4.0, 5.0]  (server-side $range)")


───────────────────────────────────────────────────────────────────────────────
  Filter: rating in [4.0, 5.0]  (server-side $range)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_01     0.6888   tech         2023   4.5     alice    True     Neural Nets & Vision
  3     doc_16     0.7415   health       2022   4.1     bob      False    Telemedicine Access
  4     doc_10     0.7522   science      2023   4.1     alice    False    AI Climate Models
  5     doc_05     0.7530   tech         2020   4.9     alice    True     Transformer Architecture



### 7.10  `$in` — OR within a field (multiple categories)

`$in` matches documents where the field value is **any** of the listed values — an OR within a single field. Use it for multi-tab views, tag-based retrieval, and permission groups.

In [17]:
results_health_science = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"category": {"$in": ["health", "science"]}}],
)
show_results(results_health_science, label='Filter: category in ["health", "science"]  ($in, 8 candidates)')


───────────────────────────────────────────────────────────────────────────────
  Filter: category in ["health", "science"]  ($in, 8 candidates)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_15     0.7386   health       2021   3.8     dave     False    Wearable Health Tech
  3     doc_16     0.7415   health       2022   4.1     bob      False    Telemedicine Access
  4     doc_10     0.7522   science      2023   4.1     alice    False    AI Climate Models
  5     doc_14     0.7837   health       2023   4.4     bob      True     Mental Health Apps



### 7.11  `$in` — multiple authors

Fetch articles by alice or bob. `$in` on string fields is useful for team dashboards, attribution views, or access control where a user belongs to multiple permission groups.

In [18]:
results_alice_bob = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"author": {"$in": ["alice", "bob"]}}],
)
show_results(results_alice_bob, label='Filter: author in ["alice", "bob"]  ($in, 11 candidates)')


───────────────────────────────────────────────────────────────────────────────
  Filter: author in ["alice", "bob"]  ($in, 11 candidates)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_01     0.6888   tech         2023   4.5     alice    True     Neural Nets & Vision
  3     doc_16     0.7415   health       2022   4.1     bob      False    Telemedicine Access
  4     doc_10     0.7522   science      2023   4.1     alice    False    AI Climate Models
  5     doc_05     0.7530   tech         2020   4.9     alice    True     Transformer Architecture



### 7.12  `$in` — specific year cohorts (numeric)

`$in` on numeric fields selects exact discrete values. Unlike `$range [2022, 2024]` which includes 2023, `$in [2022, 2024]` skips it entirely. Useful for fiscal year comparisons or versioned snapshots.

In [19]:
results_selected_years = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"year": {"$in": [2022, 2024]}}],
)
show_results(results_selected_years, label="Filter: year in [2022, 2024]  ($in numeric, 9 candidates)")


───────────────────────────────────────────────────────────────────────────────
  Filter: year in [2022, 2024]  ($in numeric, 9 candidates)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_16     0.7415   health       2022   4.1     bob      False    Telemedicine Access
  3     doc_06     0.7612   tech         2022   4.2     carol    True     Graph Neural Networks
  4     doc_07     0.8164   tech         2024   4.5     alice    True     Self-Driving Cars
  5     doc_11     0.8443   science      2022   4.0     carol    False    Precision Agriculture



### 7.13  AND: `$eq` + `$range` + `$range`

Combine an exact match with two numeric ranges across three fields: tech articles from 2022–2024 with rating ≥ 4.3. The server evaluates all conditions before any vector computation begins.

In [20]:
results_tech_recent_quality = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[
        {"category": {"$eq": "tech"}},
        {"year":     {"$range": [2022, 2024]}},
        {"rating":   {"$range": [4.3, 5.0]}},
    ],
)
show_results(
    results_tech_recent_quality,
    label='category=="tech" AND year∈[2022,2024] AND rating∈[4.3,5.0]'
)


───────────────────────────────────────────────────────────────────────────────
  category=="tech" AND year∈[2022,2024] AND rating∈[4.3,5.0]
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_01     0.6888   tech         2023   4.5     alice    True     Neural Nets & Vision
  2     doc_07     0.8164   tech         2024   4.5     alice    True     Self-Driving Cars
  3     doc_03     0.8747   tech         2023   4.3     carol    False    Vector Databases
  4     doc_02     0.9531   tech         2024   4.8     alice    True     Quantum Computing



### 7.14  Three-way AND: `$in` + `$range` + `$eq`

Mix all three operators: health-or-science (`$in`), published 2021–2023 (`$range`), and premium only (`$eq`). This mirrors a real production pattern — *show premium STEM content from the last few years.*

In [21]:
results_stem_premium = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[
        {"category": {"$in":   ["health", "science"]}},
        {"year":     {"$range": [2021, 2023]}},
        {"premium":  {"$eq":   True}},
    ],
)
show_results(
    results_stem_premium,
    label='category∈["health","science"] AND year∈[2021,2023] AND premium==True'
)


───────────────────────────────────────────────────────────────────────────────
  category∈["health","science"] AND year∈[2021,2023] AND premium==True
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_13     0.5364   health       2022   4.2     bob      True     ML in Healthcare
  2     doc_14     0.7837   health       2023   4.4     bob      True     Mental Health Apps
  3     doc_09     0.8833   science      2022   4.7     bob      True     CRISPR Gene Editing



### 7.15  `$range` — tight window (top-tier articles only)

A narrow `rating ∈ [4.5, 5.0]` window acts as a precision gate. Only documents with top ratings qualify. The similarity model ranks semantically within that set — a high rating does not guarantee a high semantic match to the query. Documents that pass the rating gate but are topically distant from the query will rank last.


In [22]:
results_top_tier = index.query(
    vector=query_vec,
    top_k=TOP_K,
    filter=[{"rating": {"$range": [4.5, 5.0]}}],
)
show_results(results_top_tier, label="Filter: rating in [4.5, 5.0]  (6 candidates)")


───────────────────────────────────────────────────────────────────────────────
  Filter: rating in [4.5, 5.0]  (6 candidates)
  Rank  ID         Score    Category     Year   Rating  Author   Premium  Title
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1     doc_01     0.6888   tech         2023   4.5     alice    True     Neural Nets & Vision
  2     doc_05     0.7530   tech         2020   4.9     alice    True     Transformer Architecture
  3     doc_07     0.8164   tech         2024   4.5     alice    True     Self-Driving Cars
  4     doc_09     0.8833   science      2022   4.7     bob      True     CRISPR Gene Editing
  5     doc_18     0.9488   business     2024   4.6     dave     True     Clean Energy VC



---

## Summary

**Congratulations, you have just built a fully filtered semantic search engine!** You ran 15 real queries against the same index and saw exactly how `$eq`, `$in`, and `$range` change the candidate pool before ranking begins.

Key things to remember:

- Declare every filterable field in the `filter` dict **at upsert time** — you cannot query a field that was not declared.
- `$eq` for exact match, `$in` for OR within a field, `$range` for inclusive numeric ranges. All evaluated server-side.
- Multiple filters in the list are always **ANDed**.
- Filters are eligibility gates, not ranking signals. A highly rated but irrelevant document that passes the filter still ranks at the bottom.

## Next Steps

- **Add sparse retrieval** — pair these filters with SPLADE for better keyword coverage when users type exact terms → [Hybrid Search with SPLADE](./hybrid_splade.ipynb)
- **Benchmark precision** — see whether int16 quantization preserves your filtered results at half the memory cost → [Precision Benchmark](./precision_benchmark.ipynb)

---

## Under the Hood: How Filters Execute

Endee's filter system uses a **pre-filter + adaptive search** strategy. Understanding this explains both the performance characteristics and the occasional score differences you see between filtered and unfiltered runs.

### Execution pipeline

```
1. Filter analysis
   └─ Each condition estimates its cardinality (how many IDs match)

2. Cheapest-first execution
   └─ Conditions evaluated in ascending cardinality order
   └─ Stops early if intermediate result becomes empty

3. Build a Roaring bitmap of matching document IDs

4. Adaptive vector search
   ├─ < 1,000 matching IDs → bypass HNSW, compute exact distances (brute force)
   └─ ≥ 1,000 matching IDs → pass bitmap to HNSW's searchKnn via filter functor
```

### Storage per operator

| Filter type | Storage | Lookup |
|-------------|---------|--------|
| `$eq` on string / bool | Inverted index — key: `field:value`, value: Roaring bitmap | Direct key lookup |
| `$in` on string | Same inverted index | Multiple key lookups → bitmap union |
| `$eq` on number | Hybrid bucket (LMDB B+ Tree) | Point range query |
| `$in` on number | Hybrid bucket | One point query per value → bitmap union |
| `$range` on number | Hybrid bucket | Cursor scan with summary-bitmap fast path for interior buckets |

The Roaring bitmap is the universal result type — all conditions return bitmaps that are intersected (AND) or unioned (`$in`) at query time, before vector search begins.

---

## A Note on Score Values

You may notice that the numeric score for the same document differs between filtered and unfiltered results. This is expected behaviour, not a bug.

Endee uses **two different search paths** depending on how many documents pass the filter:

| Search path | When used | Score meaning | Lower is? |
|-------------|-----------|---------------|-----------|
| **HNSW** | Filter pool ≥ 1,000 IDs | Cosine **similarity** | Worse |
| **Brute-force** | Filter pool < 1,000 IDs | Cosine **distance** | Better |

In both cases the **rank ordering is correct**. The numeric values just live on opposite scales depending on which path was taken.

When a filter reduces the pool to a small number of candidates (as in most of the examples in this notebook), brute-force kicks in and you will see lower scores for semantically closer documents.

**Practical implication:** do not compare raw `"similarity"` scores across filtered and unfiltered queries. Use rank positions instead, or the `"distance"` field (which is always the lower-is-better value).


---

## Real-World Patterns

These patterns compose directly from the operators above. They are what production systems actually need.

| Use case | Filter pattern |
|----------|----------------|
| **Multi-tenancy** | `{"tenant_id": {"$eq": user.org_id}}` — scope every query to the calling org without changes to the embedding model |
| **Content tiers** | `{"premium": {"$eq": True}}` — free and paid users run the same query; the filter determines the candidate pool |
| **Recency gates** | `{"year": {"$range": [current_year - 1, current_year]}}` — surface fresh content without penalising older documents in the similarity score |
| **Quality floors** | `{"rating": {"$range": [4.0, 5.0]}}` — the similarity model finds the most relevant; the quality gate enforces a minimum bar |
| **Multi-topic view** | `{"category": {"$in": ["health", "science"]}}` — one index serves multiple tabs, no duplicate data |
| **Team dashboards** | `{"author": {"$in": team_member_ids}}` — show content from a specific team, ordered by relevance to the current question |
| **Fiscal year cohorts** | `{"year": {"$in": [2022, 2023]}}` — compare specific years without affecting others in the index |

---

## Operator Reference

```python
# $eq — exact match (string, bool, int)
filter=[{"category": {"$eq": "tech"}}]
filter=[{"premium":  {"$eq": True}}]
filter=[{"year":     {"$eq": 2023}}]

# $in — list membership / OR within field (string or numeric)
filter=[{"category": {"$in": ["health", "science"]}}]
filter=[{"author":   {"$in": ["alice", "bob", "carol"]}}]
filter=[{"year":     {"$in": [2022, 2023, 2024]}}]

# $range — inclusive numeric range [start, end] (int or float)
filter=[{"year":   {"$range": [2022, 2024]}}]
filter=[{"rating": {"$range": [4.0, 5.0]}}]

# AND — multiple filters in the list (all must match)
filter=[
    {"category": {"$eq":    "tech"}},
    {"year":     {"$range": [2022, 2024]}},
    {"rating":   {"$range": [4.5, 5.0]}},
    {"premium":  {"$eq":    True}},
]
```

All operators are evaluated server-side before vector ranking.  
No `top_k=len(DOCUMENTS)` + Python post-filter needed.

---

## Key Takeaways

- **`$eq`, `$in`, `$range` cover almost every real-world use case.** Exact match for categories and flags, list membership for multi-value OR, numeric range for dates, scores, prices, and ages.

- **All three operators are server-side.** Endee evaluates them against the filter index before any vector computation begins.

- **AND is the only multi-filter logic.** Multiple filters in the `filter` list are always ANDed. OR across *different* fields requires separate queries and client-side merging; OR *within* a field is what `$in` is for.

- **Declare everything at upsert time.** Fields missing from the `filter` dict at upsert cannot be queried later. Think of it as a schema declaration.

- **Filters are eligibility gates, not ranking signals.** A highly-rated irrelevant document that passes the filter will still rank at the bottom. A highly-relevant document that fails the filter will not appear at all. If you want rating or recency to influence rank, that requires re-ranking with a combined score — not a filter.

---

*All results from Endee local mode, `all-MiniLM-L6-v2` (384-dim cosine), 20-document corpus with 5 metadata fields.*  
*Query: `"AI applications in healthcare and medicine"`. Filter operators: `$eq`, `$in`, `$range`.*

**Next steps:**
- [Hybrid Search with SPLADE](./hybrid_splade.ipynb) — pair filters with SPLADE sparse retrieval for better keyword coverage
- [Precision Benchmark](./precision_benchmark.ipynb) — test whether int16 quantization preserves filtered results at half the memory cost